# Sketchbook

Catalogs for exploratory analysis:

- `finalcat`: the complete original FINALCAT.
- `cosmos_masters`: field master catalogs keyed by field number (1–4).
- `cosmos1_master`, ..., `cosmos4_master`: convenient aliases for those tables.
- `matched_all_cosmos`: the saved COSMOS1–4 union containing only FINALCAT sources matched to 7DT in at least one field; one row per `finalcat_idx`.

Field master catalogs also contain `7dt_only` and `finalcat_only` rows. Field-local `master_id` values are distinct from the global IDs in the union catalog. Use `finalcat_idx` to associate reference sources across catalogs; FINALCAT `ID` strings are not guaranteed to be unique.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table, MaskedColumn, join, vstack
from astropy.wcs import WCS
from astropy.visualization import ImageNormalize, ZScaleInterval
from IPython.display import display

%matplotlib inline


In [3]:
# Find the project from either the repository root or a notebook subdirectory.
working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (directory for directory in (working_dir, *working_dir.parents)
     if (directory / 'COSMOS_data' / 'COSMOS_finalcat.fits').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the 7DTonCOSMOS project.')

COSMOS_FIELDS = (1, 2, 3, 4)
BANDS_7DT = tuple(f'm{wavelength}' for wavelength in range(400, 900, 25))
FINALCAT_PATH = PROJECT_ROOT / 'COSMOS_data/COSMOS_finalcat.fits'
COSMOS_MASTER_PATHS = {
    field: PROJECT_ROOT / f'output/COSMOS{field}/singlephot/cosmos{field}_master_catalog.fits'
    for field in COSMOS_FIELDS
}
# The spelling "Fianlcat" is part of the existing production filename.
MATCHED_ALL_COSMOS_PATH = (
    PROJECT_ROOT / 'output/singlephot'
    / 'master_catalog_MatchedWithFianlcat_OverallCosmosField.fits'
)


In [4]:
def read_matched_catalog(path):
    """Restore the union catalog's exact masks, including Boolean/string masks.

    Notebook 2.3 stores masks in a separate packed COLUMN_MASKS extension.
    A normal Table.read alone does not fully restore that representation.
    """
    catalog = Table.read(path, hdu='CATALOG')
    with fits.open(path, memmap=False) as hdus:
        mask_hdu = hdus['COLUMN_MASKS']
        masks = np.unpackbits(
            np.asarray(mask_hdu.data), axis=1,
            bitorder=mask_hdu.header['BITORDER'],
        )[:, :int(mask_hdu.header['NCOLS'])].astype(bool)
    for column_index, name in enumerate(catalog.colnames):
        source = catalog[name]
        catalog.replace_column(
            name,
            MaskedColumn(
                np.asarray(source), name=name, mask=masks[:, column_index],
                unit=source.unit, description=source.description,
            ),
        )
    return catalog


def numeric_column(catalog, name, fill_value=np.nan):
    """Return a float array with masked entries replaced for NumPy selections.

    This helper does not remove numeric sentinel values such as RM=-99 or RM=0.
    Valid FINALCAT magnitudes require an additional finite, 0 < RM < 50 cut.
    """
    return np.asarray(np.ma.filled(np.ma.asarray(catalog[name], dtype=float), fill_value))


In [5]:
# Load existing catalogs without performing new matching or changing files.
finalcat = Table.read(FINALCAT_PATH, hdu=1)
cosmos_masters = {
    field: Table.read(path, hdu=1)
    for field, path in COSMOS_MASTER_PATHS.items()
}
cosmos1_master = cosmos_masters[1]
cosmos2_master = cosmos_masters[2]
cosmos3_master = cosmos_masters[3]
cosmos4_master = cosmos_masters[4]
matched_all_cosmos = read_matched_catalog(MATCHED_ALL_COSMOS_PATH)

catalog_summary = Table(
    rows=[('finalcat', len(finalcat), len(finalcat.colnames))]
    + [(f'cosmos{field}_master', len(catalog), len(catalog.colnames))
       for field, catalog in cosmos_masters.items()]
    + [('matched_all_cosmos', len(matched_all_cosmos), len(matched_all_cosmos.colnames))],
    names=['variable', 'n_sources', 'n_columns'],
)
display(catalog_summary)


variable,n_sources,n_columns
str18,int64,int64
finalcat,126280,158
cosmos1_master,88361,301
cosmos2_master,93083,301
cosmos3_master,82144,301
cosmos4_master,88378,301
matched_all_cosmos,9326,775


## Scratch cells

For example, select valid `RM < 21` reference sources with:

```python
rm = numeric_column(finalcat, 'RM')
bright_finalcat = finalcat[np.isfinite(rm) & (rm > 0) & (rm < 21)]
```

The corresponding column in `matched_all_cosmos` is `finalcat_RM`.


In [10]:
x = finalcat[(finalcat['RM']<21)] #& (finalcat['RM']>0) ]
print(f'finalcat 0< RM <= 20 sources {len(x)}')

finalcat 0< RM <= 20 sources 9469


In [11]:
# Count FINALCAT rows with usable RM and the catalog's selected spectroscopic redshift.
# Z_TOT_ZSOURCE is retained as provenance; checking it also excludes the "NN" no-source sentinel.
finalcat_rm = numeric_column(finalcat, 'RM')
finalcat_spec_z = numeric_column(finalcat, 'Z_TOT_Z')
finalcat_spec_source = np.char.strip(np.asarray(finalcat['Z_TOT_ZSOURCE']).astype(str))

finalcat_rm_le21 = np.isfinite(finalcat_rm) & (finalcat_rm > 0) & (finalcat_rm <= 21)
finalcat_spec_z_usable = np.isfinite(finalcat_spec_z) & (finalcat_spec_z >= 0) & (finalcat_spec_source != 'NN')
finalcat_rm_le21_with_spec_z = finalcat_rm_le21 & finalcat_spec_z_usable

print(f'FINALCAT sources with 0 < RM <= 21: {int(finalcat_rm_le21.sum()):,}')
print(f'With usable spec-z (Z_TOT_Z >= 0; Z_TOT_ZSOURCE != NN): '
      f'{int(finalcat_rm_le21_with_spec_z.sum()):,}')
print(f'Without usable spec-z: {int((finalcat_rm_le21 & ~finalcat_spec_z_usable).sum()):,}')


FINALCAT sources with 0 < RM <= 21: 6,991
With usable spec-z (Z_TOT_Z >= 0; Z_TOT_ZSOURCE != NN): 5,874
Without usable spec-z: 1,117


## FINALCAT: spec-z usable sources with RM ≤ 21

Count rows with finite `0 < RM ≤ 21` and a valid combined reference redshift in `Z_TOT_Z` (finite and `>= 0`). `Z_TOT_ZSOURCE` records which spectroscopic reference supplied that redshift; rows whose source is `NN` are excluded. The component redshift columns (`Z_ZCOS_Z`, `Z_HCOS_Z`, `Z_DCOS_Z`, `M13_ZSPEC`, `Z_SDSS_Z`, `Z_JCOS_Z`, and literature survey columns) feed into the catalog's combined redshift, so this count uses the catalog's chosen value once per FINALCAT row instead of summing possibly overlapping source catalogs. This is a catalog-availability criterion; it does not impose a quality flag or uncertainty threshold.
